## 1, Import libraries

In [1]:
import json
import sys
from pathlib import Path
from collections import Counter
import ast
import re
from z3 import z3 as _z3

# Resolve project root & import normalizer
project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_prep.simple_universal_normalizer import SimpleUniversalNormalizer

# Paths
INPUT_FILE  = project_root / "data" / "raw" / "Logic_Based_Educational_Queries.json"
OUTPUT_DIR  = project_root / "data" / "processed"
OUTPUT_FILE = OUTPUT_DIR / "Logic_Based_Educational_Queries_Normalization.json"

ORIGINAL_FILE = project_root / "data" / "processed" / "Logic_Based_Educational_Queries_Normalization.json"
CONCLUSION_FILE = project_root / "data" / "processed" / "conclusion_fol.json"


if not INPUT_FILE.exists():
    print(f"Don't find input file: {INPUT_FILE}")

## 2, Normalization premises-FOL field

In [2]:
# Load raw data
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} records from:\n  {INPUT_FILE}")

# Normalize premises-FOL, keep everything else unchanged
normalizer = SimpleUniversalNormalizer()

total_fol   = 0
success     = 0
failed      = 0
format_dist = Counter()
error_log   = []
normalized_data = []

for i, record in enumerate(data):
    if (i + 1) % 50 == 0:
        print(f"  Processing record {i + 1}/{len(data)} \u2026")

    new_record = record.copy()
    premises_fol = record.get("premises-FOL", [])
    normalized_premises = []

    for j, fol in enumerate(premises_fol):
        total_fol += 1

        # classify original format (informational)
        if "\u2200" in fol or "\u2203" in fol:
            format_dist["unicode_quantifier"] += 1
        elif "ForAll" in fol or "Exists" in fol:
            format_dist["text_quantifier"] += 1
        else:
            format_dist["ground_atom"] += 1

        # normalize
        try:
            normalized = normalizer.normalize(fol)
            normalized_premises.append(normalized)
            success += 1
        except Exception as e:
            failed += 1
            normalized_premises.append(fol)
            if len(error_log) < 20:
                error_log.append({
                    "record": i + 1,
                    "premise": j + 1,
                    "fol": fol,
                    "error": str(e),
                })
                print(f"  \u2717 Record {i+1}, Premise {j+1}: {e}")

    new_record["premises-FOL"] = normalized_premises
    normalized_data.append(new_record)

# Save normalized dataset
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(normalized_data, f, ensure_ascii=False, indent=2)

# Print summary
print("\n" + "=" * 70)
print("  NORMALIZATION SUMMARY")
print("=" * 70)
print(f"    Total records        : {len(data)}")
print(f"    Total FOL strings    : {total_fol}")
pct_ok = (success / total_fol * 100) if total_fol else 0
print(f"    Normalized OK        : {success}  ({pct_ok:.1f}%)")
print(f"    Failed (kept orig.)  : {failed}")
print(f"    Original format distribution:")
for fmt, cnt in format_dist.most_common():
    print(f"        {fmt:25s}: {cnt:5d}  ({cnt/total_fol*100:.1f}%)")
print(f"    Saved to: {OUTPUT_FILE}")

Loaded 411 records from:
  d:\Education\exact_2026\data\raw\Logic_Based_Educational_Queries.json
  Processing record 50/411 …
  Processing record 100/411 …
  Processing record 150/411 …
  Processing record 200/411 …
  Processing record 250/411 …
  Processing record 300/411 …
  Processing record 350/411 …
  Processing record 400/411 …

  NORMALIZATION SUMMARY
    Total records        : 411
    Total FOL strings    : 4470
    Normalized OK        : 4470  (100.0%)
    Failed (kept orig.)  : 0
    Original format distribution:
        unicode_quantifier       :  2344  (52.4%)
        text_quantifier          :  1887  (42.2%)
        ground_atom              :   239  (5.3%)
    Saved to: d:\Education\exact_2026\data\processed\Logic_Based_Educational_Queries_Normalization.json


In [3]:
# Custom Iff operator for Z3
def Iff(a, b):
    return a == b

class FOLTypeInferrer:
    def __init__(self):
        self.var_types = {}  # name -> "Bool" or "Real"
        self.func_types = {} # name -> (arity, return_type)
        self.math_e_detected = False

    def infer(self, node, expected_type=None):
        if node is None:
            return

        if isinstance(node, ast.Expression):
            self.infer(node.body, expected_type)

        elif isinstance(node, ast.Name):
            if node.id in {"ForAll", "Exists", "Implies", "And", "Or", "Not", "Iff"}:
                return
            if expected_type:
                if node.id in self.var_types and self.var_types[node.id] == "Real":
                    pass
                else:
                    self.var_types[node.id] = expected_type

        elif isinstance(node, ast.Constant):
            pass

        elif isinstance(node, ast.Call):
            if isinstance(node.func, ast.Name):
                func_name = node.func.id
                if func_name in {"And", "Or", "Not", "Implies", "Iff"}:
                    for arg in node.args:
                        self.infer(arg, "Bool")
                elif func_name in {"ForAll", "Exists"}:
                    if len(node.args) > 0:
                        self.infer(node.args[0], "Real")
                    if len(node.args) > 1:
                        self.infer(node.args[1], "Bool")
                else:
                    ret_type = expected_type if expected_type else "Bool"
                    arity = len(node.args)
                    
                    if func_name in self.func_types:
                        existing_arity, existing_ret = self.func_types[func_name]
                        if existing_ret == "Real":
                            ret_type = "Real"
                    
                    self.func_types[func_name] = (arity, ret_type)
                    
                    for arg in node.args:
                        self.infer(arg, "Real")

        elif isinstance(node, ast.Compare):
            self.infer(node.left, "Real")
            for comp in node.comparators:
                self.infer(comp, "Real")

        elif isinstance(node, ast.BinOp):
            if isinstance(node.op, ast.Pow) and isinstance(node.left, ast.Name) and node.left.id == 'e':
                self.math_e_detected = True
            self.infer(node.left, "Real")
            self.infer(node.right, "Real")

        elif isinstance(node, ast.UnaryOp):
            if isinstance(node.op, (ast.USub, ast.UAdd)):
                self.infer(node.operand, "Real")
            else:
                self.infer(node.operand, "Bool")

        elif isinstance(node, ast.List):
            for elt in node.elts:
                self.infer(elt, expected_type)

def test_all_fol_with_z3():
    # Resolve project root
    project_root = Path.cwd()
    if not (project_root / "data").exists():
        project_root = project_root.parent
        
    with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    print(f"Loading {len(data)} records from: {OUTPUT_FILE}\n")
    
    UniversalSort = _z3.RealSort()
    
    total = 0
    success_count = 0
    fail_count = 0
    failures = []
    error_categories = Counter()
    
    for i, record in enumerate(data):
        for j, fol in enumerate(record.get("premises-FOL", [])):
            total += 1
            
            # Step 1: Preprocess the FOL string for Z3 and Python eval
            fol_clean = fol
            
            # Change Text to str_lit_N for safe variable naming
            string_literals = re.findall(r"'([^']*)'", fol_clean)
            string_map = {}
            for idx_lit, s_lit in enumerate(string_literals):
                var_name = f"str_lit_{idx_lit}"
                fol_clean = fol_clean.replace(f"'{s_lit}'", var_name)
                string_map[var_name] = s_lit
                
            # Change character '^' to '**' for exponentiation
            fol_clean = fol_clean.replace("^", "**")
            
            # Step 2: Run the AST-based type inference
            inferrer = FOLTypeInferrer()
            try:
                tree = ast.parse(fol_clean, mode='eval')
                inferrer.infer(tree)
            except Exception as e:
                fail_count += 1
                err_msg = f"Lỗi cú pháp Python AST: {str(e)}"
                error_categories[err_msg] += 1
                if len(failures) < 15:
                    failures.append((i+1, j+1, fol, err_msg))
                continue
                
            # Step 3: Create a namespace for Z3 execution
            ns = {
                "ForAll": _z3.ForAll,
                "Exists": _z3.Exists,
                "Implies": _z3.Implies,
                "And": _z3.And,
                "Or": _z3.Or,
                "Not": _z3.Not,
                "Iff": Iff,
            }
            
            # Register all variable names found in the AST, excluding logical operators
            all_names = set()
            for node in ast.walk(ast.parse(fol_clean)):
                if isinstance(node, ast.Name):
                    if node.id not in {"ForAll", "Exists", "Implies", "And", "Or", "Not", "Iff"}:
                        all_names.add(node.id)
                        
            for name in all_names:
                if name in ns:
                    continue
                # Check for special constants and numeric literals
                if name == 'e' and inferrer.math_e_detected:
                    ns[name] = _z3.RealVal(2.71828)
                elif name.lower() == 'pi':
                    ns[name] = _z3.RealVal(3.14159)
                elif name.isdigit():
                    ns[name] = _z3.RealVal(int(name))
                else:
                    var_type = inferrer.var_types.get(name, "Real")
                    mapped_name = string_map.get(name, name)
                    if var_type == "Bool":
                        ns[name] = _z3.Bool(mapped_name)
                    else:
                        ns[name] = _z3.Real(mapped_name)
                        
            # Register functions and predicates
            for func_name, (arity, ret_type) in inferrer.func_types.items():
                if arity == 0:
                    if ret_type == "Bool":
                        ns[func_name] = lambda fn=func_name: _z3.Bool(fn)
                    else:
                        ns[func_name] = lambda fn=func_name: _z3.Real(fn)
                else:
                    arg_sorts = [UniversalSort] * arity
                    if ret_type == "Real":
                        ns[func_name] = _z3.Function(func_name, *arg_sorts, UniversalSort)
                    else:
                        ns[func_name] = _z3.Function(func_name, *arg_sorts, _z3.BoolSort())

            # Step 4: Execute with Z3 and catch errors
            try:
                result = eval(fol_clean, {"__builtins__": {}}, ns)
                if hasattr(result, 'sexpr'):
                    success_count += 1
                else:
                    fail_count += 1
                    failures.append((i+1, j+1, fol, f"Không trả về đối tượng Z3 (trả về {type(result).__name__})"))
            except Exception as e:
                fail_count += 1
                err_msg = str(e).split('\n')[0]
                error_categories[err_msg] += 1
                if len(failures) < 15:
                    failures.append((i+1, j+1, fol, err_msg))

    # Step 5: Print summary statistics
    print("=" * 80)
    print("    RESULT OF Z3 COMPATIBILITY CHECK (AST-BASED SMART TYPE INFERENCE)")
    print("=" * 80)
    print(f"   The number of logic sentences tested: {total}")
    print(f"   Excuted successfully with Z3: {success_count}  ({success_count/total*100:.2f}%)")
    print(f"   Excuted failed with Z3: {fail_count}  ({fail_count/total*100:.2f}%)")
    print("-" * 80)
    
    if error_categories:
        print("    List of error categories:")
        for err, count in error_categories.most_common():
            print(f"    [{count:3d}] {err}")
            
    if failures:
        print("    Some representative error cases for reference:\n")
        for r, p, fol_str, err in failures[:10]:
            print(f"    - Bản ghi {r}, Câu {p}: {fol_str}")
            print(f"    Record {r}, Premise {p}: {fol_str}")
            print(f"    Error: {err}")

test_all_fol_with_z3()


Loading 411 records from: d:\Education\exact_2026\data\processed\Logic_Based_Educational_Queries_Normalization.json

    RESULT OF Z3 COMPATIBILITY CHECK (AST-BASED SMART TYPE INFERENCE)
   The number of logic sentences tested: 4470
   Excuted successfully with Z3: 4470  (100.00%)
   Excuted failed with Z3: 0  (0.00%)
--------------------------------------------------------------------------------


## Merge results conclusion_fol from LLM to original file

In [4]:
# Read the original normalized file
print(f"Loading original data from: {ORIGINAL_FILE}")
with open(ORIGINAL_FILE, "r", encoding="utf-8") as f:
    original_data = json.load(f)

# Read the conclusions file generated by LLM
print(f"Đang đọc file conclusions từ: {CONCLUSION_FILE}")
with open(CONCLUSION_FILE, "r", encoding="utf-8") as f:
    conclusions_data = json.load(f)

# Impletent merge based on ID
conclusion_lookup = {item["id"]: item["conclusion_fol"] for item in conclusions_data}

merged_count = 0
for i, record in enumerate(original_data):
    if i in conclusion_lookup:
        record["conclusion_fol"] = conclusion_lookup[i]
        merged_count += 1
    else:
        print(f"Note: Don't find conclusion_fol for record {i}")

# Save original file
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(original_data, f, ensure_ascii=False, indent=2)

print(f"Merged successfully {merged_count}/{len(original_data)} 'conclusion_fol' fields into original dataset")
print(f"Data with conclusions saved at: {OUTPUT_FILE}")

Loading original data from: d:\Education\exact_2026\data\processed\Logic_Based_Educational_Queries_Normalization.json
Đang đọc file conclusions từ: d:\Education\exact_2026\data\processed\conclusion_fol.json
Merged successfully 411/411 'conclusion_fol' fields into original dataset
Data with conclusions saved at: d:\Education\exact_2026\data\processed\Logic_Based_Educational_Queries_Normalization.json


## Check answer by Z3 Solver

In [5]:
UniversalSort = _z3.RealSort()
def coerce_to_bool(expr):
    """Gradually coerce the expression to a boolean context for Z3 compatibility"""
    if isinstance(expr, bool):
        return expr
    if hasattr(expr, 'sort'):
        sort = expr.sort()
        if sort == _z3.BoolSort():
            return expr
        elif sort == _z3.RealSort() or sort == _z3.IntSort():
            return expr != 0
    return expr
def CustomAnd(*args):
    coerced = [coerce_to_bool(x) for x in args]
    return _z3.And(*coerced)
def CustomOr(*args):
    coerced = [coerce_to_bool(x) for x in args]
    return _z3.Or(*coerced)
def CustomNot(arg):
    return _z3.Not(coerce_to_bool(arg))
def CustomImplies(a, b):
    return _z3.Implies(coerce_to_bool(a), coerce_to_bool(b))
def CustomIff(a, b):
    return coerce_to_bool(a) == coerce_to_bool(b)
def CustomForAll(vars_list, body):
    return _z3.ForAll(vars_list, coerce_to_bool(body))
def CustomExists(vars_list, body):
    return _z3.Exists(vars_list, coerce_to_bool(body))

def clean_and_register_names(fol_str, string_map):
    # Join multi-word predicates with _ (e.g., passed science assessment(x) -> passed_science_assessment(x))
    fol_clean = re.sub(r'\b([a-z]+(?:\s+[a-z]+)+)\s*\(', lambda m: m.group(1).replace(' ', '_') + '(', fol_str)
    fol_clean = fol_clean.strip().rstrip('.$')
    fol_clean = fol_clean.replace("^", "**")
    
    string_literals = re.findall(r"'([^']*)'", fol_clean)
    for idx_lit, s_lit in enumerate(string_literals):
        var_name = f'str_lit_{idx_lit}'
        fol_clean = fol_clean.replace(f"'{s_lit}'", var_name)
        string_map[var_name] = s_lit
        
    return fol_clean

class FOLASTTransformer(ast.NodeTransformer):
    def visit_Compare(self, node):
        if isinstance(node.left, ast.BinOp) and isinstance(node.left.op, (ast.BitAnd, ast.BitOr)):
            binop = node.left
            new_compare = ast.Compare(
                left=binop.right,
                ops=node.ops,
                comparators=node.comparators
            )
            ast.copy_location(new_compare, node)
            func_name = 'And' if isinstance(binop.op, ast.BitAnd) else 'Or'
            call_node = ast.Call(
                func=ast.Name(id=func_name, ctx=ast.Load()),
                args=[binop.left, new_compare],
                keywords=[]
            )
            ast.copy_location(call_node, node)
            return self.visit(call_node)
        
        self.generic_visit(node)
        return node
    def visit_BinOp(self, node):
        self.generic_visit(node)
        if isinstance(node.op, ast.Pow):
            return ast.Call(
                func=ast.Name(id='Power', ctx=ast.Load()),
                args=[node.left, node.right],
                keywords=[]
            )
        elif isinstance(node.op, ast.BitAnd):
            return ast.Call(
                func=ast.Name(id='And', ctx=ast.Load()),
                args=[node.left, node.right],
                keywords=[]
            )
        elif isinstance(node.op, ast.BitOr):
            return ast.Call(
                func=ast.Name(id='Or', ctx=ast.Load()),
                args=[node.left, node.right],
                keywords=[]
            )
        return node
    def visit_Call(self, node):
        self.generic_visit(node)
        if isinstance(node.func, ast.Name):
            func_name = node.func.id
            if func_name not in {'ForAll', 'Exists', 'Implies', 'And', 'Or', 'Not', 'Iff', 'Power'}:
                arity = len(node.args)
                node.func.id = f'{func_name}_arity_{arity}'
        return node

# Run the verification process with Z3

try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass
    
with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print("=" * 80)
print("    STARTING Z3 COMPATIBILITY CHECK FOR THE ENTIRE DATASET")
print(f"    Total records: {len(data)}")
print("=" * 80)

total_questions = 0
passed_questions = 0
failed_questions = 0
errors_log = []

for i, record in enumerate(data):
    premises = record.get('premises-FOL', [])
    questions = record.get('questions', [])
    answers = record.get('answers', [])
    conclusion_fol = record.get('conclusion_fol', [])
    
    all_fol_strings = list(premises)
    for conc in conclusion_fol:
        if isinstance(conc, dict):
            all_fol_strings.extend(conc.values())
        elif isinstance(conc, str):
            all_fol_strings.append(conc)
            
    string_map = {}
    cleaned_trees = []
    record_names = set()
    record_functions = {}
    math_e_detected = False
    function_return_sorts = {}
    
    try:
        # Step 1: Collect variable, constant, and function information through AST Walk
        for fol in all_fol_strings:
            cleaned = clean_and_register_names(fol, string_map)
            tree = ast.parse(cleaned, mode='eval')
            ast.fix_missing_locations(tree)
            transformed_tree = FOLASTTransformer().visit(tree)
            ast.fix_missing_locations(transformed_tree)
            cleaned_trees.append(transformed_tree)
            
            # Implemented AST type inference to deduce if functions return Bool or Real based on context
            for node in ast.walk(transformed_tree):
                if isinstance(node, ast.Name):
                    if node.id not in {'ForAll', 'Exists', 'Implies', 'And', 'Or', 'Not', 'Iff', 'Power'}:
                        record_names.add(node.id)
                elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
                    func_name = node.func.id
                    if func_name not in {'ForAll', 'Exists', 'Implies', 'And', 'Or', 'Not', 'Iff', 'Power'}:
                        if '_arity_' in func_name:
                            parts = func_name.split('_arity_')
                            arity = int(parts[-1])
                            record_functions[func_name] = arity
                        else:
                            record_functions[func_name] = len(node.args)
                        
                        if func_name not in function_return_sorts:
                            function_return_sorts[func_name] = _z3.BoolSort()
                    elif func_name == 'Power':
                        if isinstance(node.args[0], ast.Name) and node.args[0].id == 'e':
                            math_e_detected = True
                            
                # If a function appears in a numeric comparison context or is compared to a numeric literal
                elif isinstance(node, ast.Compare):
                    is_numeric_op = any(isinstance(op, (ast.Gt, ast.GtE, ast.Lt, ast.LtE)) for op in node.ops)
                    if not is_numeric_op:
                        all_sides = [node.left] + node.comparators
                        has_number = any(
                            isinstance(side, (ast.Num, ast.Constant)) and isinstance(getattr(side, 'n', getattr(side, 'value', None)), (int, float))
                            for side in all_sides
                        )
                        if has_number:
                            is_numeric_op = True
                            
                    if is_numeric_op:
                        for subnode in ast.walk(node):
                            if isinstance(subnode, ast.Call) and isinstance(subnode.func, ast.Name):
                                f_name = subnode.func.id
                                function_return_sorts[f_name] = UniversalSort
                                
                # If a function is involved in an arithmetic operation
                elif isinstance(node, ast.BinOp):
                    if isinstance(node.op, (ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Mod, ast.Pow)):
                        for subnode in [node.left, node.right]:
                            for call_node in ast.walk(subnode):
                                if isinstance(call_node, ast.Call) and isinstance(call_node.func, ast.Name):
                                    f_name = call_node.func.id
                                    function_return_sorts[f_name] = UniversalSort
                                    
                # If a function appears as an argument to Power
                elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id == 'Power':
                    for arg in node.args:
                        for call_node in ast.walk(arg):
                            if isinstance(call_node, ast.Call) and isinstance(call_node.func, ast.Name):
                                f_name = call_node.func.id
                                function_return_sorts[f_name] = UniversalSort
                            
        # Step 2: Set up a namespace that preserves the inferred types for Z3 execution
        ns = {
            'ForAll': CustomForAll,
            'Exists': CustomExists,
            'Implies': CustomImplies,
            'And': CustomAnd,
            'Or': CustomOr,
            'Not': CustomNot,
            'Iff': CustomIff,
            'Power': _z3.Function('Power', _z3.RealSort(), _z3.RealSort(), _z3.RealSort())
        }
        
        # Identify predicates (functions returning Bool) and functions (returning Real)
        for func_name, arity in record_functions.items():
            arg_sorts = [UniversalSort] * arity
            ret_sort = function_return_sorts.get(func_name, _z3.BoolSort())
            ns[func_name] = _z3.Function(func_name, *arg_sorts, ret_sort)
        
        # Identify constants and entities
        for name in record_names:
            if name in ns: 
                continue
            if name == 'e' and math_e_detected:
                ns[name] = _z3.RealVal(2.71828)
            elif name.lower() == 'pi':
                ns[name] = _z3.RealVal(3.14159)
            elif name.isdigit():
                ns[name] = _z3.RealVal(int(name))
            elif name in record_functions:
                continue
            else:
                mapped_name = string_map.get(name, name)
                ns[name] = _z3.Const(mapped_name, UniversalSort)
                
        # Step 3: Compile premises into Z3 expressions
        premises_exprs = []
        for fol in premises:
            cleaned = clean_and_register_names(fol, string_map)
            tree = ast.parse(cleaned, mode='eval')
            ast.fix_missing_locations(tree)
            transformed_tree = FOLASTTransformer().visit(tree)
            ast.fix_missing_locations(transformed_tree)
            code = compile(transformed_tree, filename='<ast>', mode='eval')
            res = eval(code, {'__builtins__': {}}, ns)
            premises_exprs.append(coerce_to_bool(res))
            
        # Step 4: Evaluate each question in the record
        for q_idx, (q_text, expected_ans, conc) in enumerate(zip(questions, answers, conclusion_fol)):
            total_questions += 1
            
            base_solver = _z3.Solver()
            base_solver.set("timeout", 2000)
            for prem_expr in premises_exprs:
                base_solver.add(prem_expr)
                
            predicted_ans = None
            detailed_err = None
            
            try:
                # Check the consistency of the premises first before trying to prove anything
                if base_solver.check() == _z3.unsat:
                    predicted_ans = "Error: Inconsistent Premises"
                    raise ValueError("Hệ thống giả thiết (Premises) tự mâu thuẫn lẫn nhau.")
                    
                # Case A: Multiple Choice Question with options in a dictionary
                if isinstance(conc, dict):
                    proved_options = []
                    for opt, opt_fol in conc.items():
                        cleaned_opt = clean_and_register_names(opt_fol, string_map)
                        tree = ast.parse(cleaned_opt, mode='eval')
                        ast.fix_missing_locations(tree)
                        transformed_tree = FOLASTTransformer().visit(tree)
                        ast.fix_missing_locations(transformed_tree)
                        code = compile(transformed_tree, filename='<ast>', mode='eval')
                        opt_expr = coerce_to_bool(eval(code, {'__builtins__': {}}, ns))
                        
                        # Prove opt_expr is true by showing that Not(opt_expr) leads to unsat
                        base_solver.push()
                        base_solver.add(_z3.Not(opt_expr))
                        res = base_solver.check()
                        base_solver.pop()
                        
                        if res == _z3.unsat:
                            proved_options.append(opt)
                    if len(proved_options) == 1:
                        predicted_ans = proved_options[0]
                    elif len(proved_options) > 1:
                        predicted_ans = f"Error: Multiple True Options ({', '.join(proved_options)})"
                    else:
                        predicted_ans = "Cannot Prove Any Option"
                        
                # Case B: Yes/No Question with a single conclusion string
                elif isinstance(conc, str):
                    cleaned_conc = clean_and_register_names(conc, string_map)
                    tree = ast.parse(cleaned_conc, mode='eval')
                    ast.fix_missing_locations(tree)
                    transformed_tree = FOLASTTransformer().visit(tree)
                    ast.fix_missing_locations(transformed_tree)
                    code = compile(transformed_tree, filename='<ast>', mode='eval')
                    conc_expr = coerce_to_bool(eval(code, {'__builtins__': {}}, ns))
                    
                    # Trying to prove conc_expr is true by showing that Not(conc_expr) leads to unsat
                    base_solver.push()
                    base_solver.add(_z3.Not(conc_expr))
                    res_yes = base_solver.check()
                    base_solver.pop()
                    
                    if res_yes == _z3.unsat:
                        predicted_ans = "Yes"
                    else:
                        # Trying to prove Not(conc_expr) is true by showing that conc_expr leads to unsat
                        base_solver.push()
                        base_solver.add(conc_expr)
                        res_no = base_solver.check()
                        base_solver.pop()
                        
                        if res_no == _z3.unsat:
                            predicted_ans = "No"
                        else:
                            predicted_ans = "Unknown"
                                
            except Exception as e:
                detailed_err = str(e)
                predicted_ans = "Error"
                
            # Comparing predicted answer with expected answer from dataset
            is_passed = (predicted_ans == expected_ans)
            if is_passed:
                passed_questions += 1
            else:
                failed_questions += 1
                errors_log.append({
                    "record_id": i,
                    "question_idx": q_idx + 1,
                    "question_text": q_text.split('\n')[0],
                    "expected": expected_ans,
                    "predicted": predicted_ans,
                    "error_type": "Discrepancy" if not detailed_err else "Z3 Runtime Error",
                    "details": detailed_err if detailed_err else f"Z3 deduced answer '{predicted_ans}' but dataset has '{expected_ans}'."
                })
                
    except Exception as e:
        errors_log.append({
            "record_id": i,
            "error_type": "Z3 Compilation Error",
            "details": f"Cannot compile premises into Z3 expressions. Error: {e}"
        })
        failed_questions += len(questions)
        total_questions += len(questions)
        
# Summarize results
accuracy = (passed_questions / total_questions * 100) if total_questions else 0
print("="* 80)
print("    Results of Z3 compatibility check for all questions in the dataset")
print("="* 80)
print(f"    Total questions checked       : {total_questions}")
print(f"    Correct answers (match Z3)  : {passed_questions}  ({accuracy:.2f}%)")
print(f"    Incorrect answers (mismatch with Z3) : {failed_questions}  ({100 - accuracy:.2f}%)")
print("-"* 80)

if errors_log:
    print("    List of detailed error cases for questions that failed Z3 compatibility check:")
    print("-"* 80)
    for err in errors_log:
        rec_id = err["record_id"]
        if err["error_type"] == "Discrepancy":
            print(f"    Record {rec_id}, Question {err['question_idx']}:")
            print(f"    Question: \"{err['question_text']}\"")
            print(f"    Answer in Dataset: {err['expected']}")
            print(f"    Z3 Solver's Answer: {err['predicted']}")
            print(f"    Details: {err['details']}")
        else:
            print(f"    Error system at Record {rec_id}: {err['error_type']}")
            print(f"    Details: {err['details']}")
        print("-"* 80)
else:
    print("    No errors detected. All questions are compatible with Z3 solver's deductions.")

    STARTING Z3 COMPATIBILITY CHECK FOR THE ENTIRE DATASET
    Total records: 411


C:\Users\NguyenAn\AppData\Local\Temp\ipykernel_18156\2735985063.py:172: DeprecationWarning: ast.Num is deprecated and will be removed in Python 3.14; use ast.Constant instead
  isinstance(side, (ast.Num, ast.Constant)) and isinstance(getattr(side, 'n', getattr(side, 'value', None)), (int, float))
C:\Users\NguyenAn\AppData\Local\Temp\ipykernel_18156\2735985063.py:172: DeprecationWarning: Attribute n is deprecated and will be removed in Python 3.14; use value instead
  isinstance(side, (ast.Num, ast.Constant)) and isinstance(getattr(side, 'n', getattr(side, 'value', None)), (int, float))


    Results of Z3 compatibility check for all questions in the dataset
    Total questions checked       : 808
    Correct answers (match Z3)  : 808  (100.00%)
    Incorrect answers (mismatch with Z3) : 0  (0.00%)
--------------------------------------------------------------------------------
    No errors detected. All questions are compatible with Z3 solver's deductions.
